In [2]:
pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [46]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import os
import zipfile
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

In [47]:
# Create SageMaker session
session = sagemaker.Session()

# Get AWS region
region = boto3.Session().region_name

# Get SageMaker execution role
role = sagemaker.get_execution_role()

# Create S3 client
s3 = boto3.client("s3")

print("AWS Region:", region)
print("SageMaker Execution Role:", role)

AWS Region: ap-southeast-2
SageMaker Execution Role: arn:aws:iam::526404916975:role/service-role/AmazonSageMaker-ExecutionRole-20260821T132340


In [48]:
bucket = "loan-ml-assignment"

print("S3 Bucket:", bucket)

S3 Bucket: loan-ml-assignment


In [49]:
print("Current bucket variable:")
print(bucket)

print("\nAWS region:")
print(region)

print("\nBuckets this AWS account can see:")

response = s3.list_buckets()

for b in response["Buckets"]:
    print(b["Name"])

Current bucket variable:
loan-ml-assignment

AWS region:
ap-southeast-2

Buckets this AWS account can see:
loan-ml-assignment


In [50]:
response = s3.list_objects_v2(
    Bucket=bucket
)

print("Files stored in S3:")

for item in response.get("Contents", []):
    print(item["Key"])

Files stored in S3:
output/
processed/
processed/train/train_xgb.csv
processed/validation/validation_xgb.csv
raw/
raw/playground-series-s4e10.zip
raw/sample_submission.csv
raw/test.csv
raw/train.csv


In [51]:
# Download training data from Amazon S3

s3.download_file(
    bucket,
    "raw/train.csv",
    "train.csv"
)

print("train.csv downloaded successfully from S3.")

train.csv downloaded successfully from S3.


In [52]:
# Load the dataset using pandas

train_df = pd.read_csv("train.csv")

print("Dataset shape:", train_df.shape)

train_df.head()

Dataset shape: (58645, 13)


,id,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status
0,0,37,35000,RENT,0.0,EDUCATION,B,6000,11.49,0.17,N,14,0
1,1,22,56000,OWN,6.0,MEDICAL,C,4000,13.35,0.07,N,2,0
2,2,29,28800,OWN,8.0,PERSONAL,A,6000,8.90,0.21,N,10,0
3,3,30,70000,RENT,14.0,VENTURE,B,12000,11.11,0.17,N,5,0
4,4,22,60000,RENT,2.0,MEDICAL,A,6000,6.92,0.10,N,3,0


In [53]:
print("Columns:")
print(train_df.columns.tolist())

print("\nTarget distribution:")
print(train_df["loan_status"].value_counts())

Columns:
['id', 'person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'loan_status']

Target distribution:
loan_status
0    50295
1     8350
Name: count, dtype: int64


In [66]:
# SELECT FEATURES AND TARGET


TARGET = "loan_status"

features = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]

X = train_df[features].copy()
y = train_df[TARGET].copy()

print("Target:", TARGET)

print("\nFeatures used:")
for feature in features:
    print(feature)

Target: loan_status

Features used:
person_age
person_income
person_emp_length
loan_amnt
loan_int_rate
loan_percent_income
cb_person_cred_hist_length


In [56]:
# Check missing values before cleaning

print("Missing values before cleaning:")
print(X.isnull().sum())

Missing values before cleaning:
person_age                    0
person_income                 0
person_emp_length             0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_cred_hist_length    0
dtype: int64


In [67]:
# TRAIN / VALIDATION SPLIT


from sklearn.model_selection import train_test_split

X_train, X_validation, y_train, y_validation = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_validation))

print("\nTraining features shape:")
print(X_train.shape)

print("\nValidation features shape:")
print(X_validation.shape)

Training rows: 46916
Validation rows: 11729

Training features shape:
(46916, 7)

Validation features shape:
(11729, 7)


In [68]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nValidation target distribution:")
print(y_validation.value_counts(normalize=True).round(3))

Training target distribution:
loan_status
0    0.858
1    0.142
Name: proportion, dtype: float64

Validation target distribution:
loan_status
0    0.858
1    0.142
Name: proportion, dtype: float64


## Baseline Machine Learning Model

A Random Forest classifier was trained in the Amazon SageMaker notebook environment.

The model provides a baseline for evaluating the loan approval prediction problem.

Accuracy and ROC AUC are used to evaluate model performance.

In [69]:
# BASELINE RANDOM FOREST MODEL
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

baseline_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

baseline_model.fit(
    X_train,
    y_train
)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [70]:
# Make predictions

baseline_predictions = baseline_model.predict(
    X_validation
)

baseline_probabilities = baseline_model.predict_proba(
    X_validation
)[:, 1]

In [71]:
# BASELINE MODEL RESULTS
baseline_accuracy = accuracy_score(
    y_validation,
    baseline_predictions
)

baseline_auc = roc_auc_score(
    y_validation,
    baseline_probabilities
)

print("Baseline Random Forest Results")
print("--------------------------------")
print(
    "Accuracy:",
    round(baseline_accuracy, 4)
)

print(
    "ROC AUC:",
    round(baseline_auc, 4)
)

Baseline Random Forest Results
--------------------------------
Accuracy: 0.9074
ROC AUC: 0.8975


## Hyperparameter Tuning

A small hyperparameter experiment was performed to investigate whether changing the Random Forest settings could improve model performance.
Two Random Forest hyperparameters were tested:
- `n_estimators`: the number of trees in the Random Forest
- `max_depth`: the maximum depth of each tree

Each parameter combination was evaluated using validation Accuracy and ROC AUC.

ROC AUC was used to select the best model.

In [76]:
# HYPERPARAMETER EXPERIMENT
n_estimators_values = [
    50,
    100,
    200
]

max_depth_values = [
    5,
    10,
    None
]

results = []

best_auc = 0
best_model = None
best_parameters = None

for n_estimators in n_estimators_values:

    for max_depth in max_depth_values:

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )

        # Train model
        model.fit(
            X_train,
            y_train
        )

        # Make predictions
        predictions = model.predict(
            X_validation
        )

        probabilities = model.predict_proba(
            X_validation
        )[:, 1]

        # Calculate performance
        accuracy = accuracy_score(
            y_validation,
            predictions
        )

        auc = roc_auc_score(
            y_validation,
            probabilities
        )

        # Save result
        results.append({
            "n_estimators": n_estimators,
            "max_depth": str(max_depth),
            "accuracy": accuracy,
            "roc_auc": auc
        })

        # Keep the best model
        if auc > best_auc:

            best_auc = auc

            best_model = model

            best_parameters = {
                "n_estimators": n_estimators,
                "max_depth": max_depth
            }

print("Hyperparameter experiment completed.")

Hyperparameter experiment completed.


In [77]:
# Convert results into a DataFrame

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "roc_auc",
    ascending=False
)

results_df

,n_estimators,max_depth,accuracy,roc_auc
1,50,10,0.913548,0.911738
4,100,10,0.912354,0.911418
7,200,10,0.913548,0.910545
8,200,None,0.912951,0.909960
5,100,None,0.912525,0.908858
2,50,None,0.911416,0.903779
6,200,5,0.907068,0.897550
3,100,5,0.907409,0.897468
0,50,5,0.908176,0.895811


In [79]:
# BEST MODEL

print("Best Random Forest Parameters")
print("-----------------------------")

print(
    "n_estimators:",
    best_parameters["n_estimators"]
)

print(
    "max_depth:",
    best_parameters["max_depth"]
)

print(
    "Best ROC AUC:",
    round(best_auc, 4)
)

Best Random Forest Parameters
-----------------------------
n_estimators: 50
max_depth: 10
Best ROC AUC: 0.9117


In [75]:
print("Model Comparison")
print("----------------")

print(
    "Baseline Accuracy:",
    round(baseline_accuracy, 4)
)

print(
    "Baseline ROC AUC:",
    round(baseline_auc, 4)
)

print()

print(
    "Best Tuned ROC AUC:",
    round(best_auc, 4)
)

Model Comparison
----------------
Baseline Accuracy: 0.9074
Baseline ROC AUC: 0.8975

Best Tuned ROC AUC: 0.9117


### Hyperparameter Tuning Result

The Random Forest model was tested using different values for `n_estimators` and `max_depth`.

The best parameter combination was selected using validation ROC AUC.

The experiment demonstrates that model parameters can be systematically compared within the SageMaker notebook environment rather than selecting model settings without evaluation.